# AI Language Translation Tool

**Project:** Language Translation  
**Author:** Saabeer  

---

### 📌 Project Overview
This notebook demonstrates the step-by-step development and testing of a modern **AI Language Translation Tool** featuring multi-language translation, speech synthesis (TTS), live word/character counters, and an iOS-inspired user interface.

#### 🎯 Objectives:
1. Allow users to enter text to translate.
2. Select a source language (or use Auto-Detect).
3. Select a target language from 15+ supported languages.
4. Process translation requests using `deep-translator` with error handling.
5. Provide character and word counting, swap languages, copy to clipboard, and Text-to-Speech (TTS).
6. Build an interactive iOS-styled web UI using **Gradio** that runs seamlessly on Google Colab and local machines.

## 1. Environment Setup & Runtime Inspection
Let us first inspect our Python runtime environment, OS platform, and version to ensure compatibility.

In [1]:
import sys
import platform

print("Python Version:", sys.version)
print("Operating System Platform:", platform.platform())

Python Version: 3.13.15
Operating System Platform: Linux-6.6.122+-x86_64-with-glibc2.39


## 2. Package Installation
We install the core required libraries:
- `gradio`: For creating the interactive user interface.
- `deep-translator`: Python library to translate text using translation services (e.g. GoogleTranslator, MyMemoryTranslator).
- `gTTS`: Google Text-to-Speech library to convert translated text to audio.

In [2]:
!pip install --quiet gradio deep-translator gTTS

## 3. Import Libraries
We import all the required modules and verify their availability.

In [3]:
from typing import Tuple, Dict
import gradio as gr
from deep_translator import GoogleTranslator, MyMemoryTranslator
from deep_translator.exceptions import (
    TooManyRequests,
    LanguageNotSupportedException,
    TranslationNotFound,
    RequestError
)
from gtts import gTTS
import tempfile
import os

print("✅ Gradio version:", gr.__version__)
print("✅ All translation and speech libraries imported successfully.")

✅ Gradio version: 6.27.0
✅ All translation and speech libraries imported successfully.


## 4. Language Configuration
We define our supported languages, including major Indian languages (Telugu, Hindi, Tamil, Kannada, Malayalam, Marathi, Bengali, Gujarati) and international languages (English, Spanish, French, German, Japanese, Chinese, Arabic).

Each language is mapped to its standard ISO 639-1 code.

In [4]:
# Supported target languages with their ISO 639-1 language codes
SUPPORTED_LANGUAGES: Dict[str, str] = {
    "English": "en",
    "Telugu": "te",
    "Hindi": "hi",
    "Tamil": "ta",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Marathi": "mr",
    "Bengali": "bn",
    "Gujarati": "gu",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Japanese": "ja",
    "Chinese": "zh-CN",
    "Arabic": "ar"
}

# Source languages include 'Auto Detect'
SOURCE_LANGUAGES: Dict[str, str] = {
    "Auto Detect": "auto",
    **SUPPORTED_LANGUAGES
}

# Target languages must be an explicit language
TARGET_LANGUAGES: Dict[str, str] = SUPPORTED_LANGUAGES

# Secondary mapping for MyMemoryTranslator fallback (handles IP rate limits)
_MYMEMORY_CODE_MAP: Dict[str, str] = {
    "auto": "autodetect",
    "en": "en-GB",
    "te": "te-IN",
    "hi": "hi-IN",
    "ta": "ta-IN",
    "kn": "kn-IN",
    "ml": "ml-IN",
    "mr": "mr-IN",
    "bn": "bn-IN",
    "gu": "gu-IN",
    "es": "es-ES",
    "fr": "fr-FR",
    "de": "de-DE",
    "ja": "ja-JP",
    "zh-CN": "zh-CN",
    "ar": "ar-SA"
}

MAX_CHAR_LIMIT = 5000

print(f"Configured {len(SOURCE_LANGUAGES)} source options and {len(TARGET_LANGUAGES)} target languages.")

Configured 16 source options and 15 target languages.


## 5. Helper & Translation Functions
We implement modular functions:
1. `get_language_code`: Safely resolves language names to ISO codes.
2. `count_text_stats`: Computes character and word counts.
3. `translate_text`: Validates input, executes translation via `GoogleTranslator`, and includes automatic fallback to `MyMemoryTranslator` if Google rate-limits (HTTP 429).

In [5]:
def get_language_code(language_name_or_code: str, is_source: bool = False) -> str:
    """
    Converts a language name (e.g. 'Telugu') or code ('te') into a valid ISO code.
    """
    if not language_name_or_code:
        raise ValueError("Language selection cannot be empty.")

    clean_name = language_name_or_code.strip()
    valid_codes = set(SOURCE_LANGUAGES.values()) if is_source else set(TARGET_LANGUAGES.values())
    
    if clean_name.lower() in [c.lower() for c in valid_codes]:
        return clean_name

    lang_dict = SOURCE_LANGUAGES if is_source else TARGET_LANGUAGES
    for name, code in lang_dict.items():
        if name.lower() == clean_name.lower():
            return code

    raise ValueError(f"Unsupported language: '{language_name_or_code}'.")


def count_text_stats(text: str) -> Tuple[int, int]:
    """
    Returns (character_count, word_count) for the provided text.
    """
    if not text:
        return 0, 0
    return len(text), len(text.split())


def translate_text(
    text: str,
    source_language: str = "Auto Detect",
    target_language: str = "Telugu"
) -> str:
    """
    Translates text with full validation and exception handling.
    """
    # 1. Empty or whitespace check
    if text is None or not text.strip():
        return ""

    clean_text = text.strip()

    # 2. Maximum character check
    if len(clean_text) > MAX_CHAR_LIMIT:
        return f"Error: Text exceeds maximum limit of {MAX_CHAR_LIMIT} characters."

    # 3. Resolve language codes
    try:
        src_code = get_language_code(source_language, is_source=True)
        tgt_code = get_language_code(target_language, is_source=False)
    except ValueError as err:
        return f"Language Error: {str(err)}"

    # 4. Identity translation check
    if src_code != "auto" and src_code.lower() == tgt_code.lower():
        return clean_text

    # 5. Translation execution with fallback
    try:
        translator = GoogleTranslator(source=src_code, target=tgt_code)
        result = translator.translate(clean_text)
        return result if result is not None else ""
    except TooManyRequests:
        # Fallback to MyMemoryTranslator if rate limited
        try:
            mem_src = _MYMEMORY_CODE_MAP.get(src_code, src_code)
            mem_tgt = _MYMEMORY_CODE_MAP.get(tgt_code, tgt_code)
            fallback = MyMemoryTranslator(source=mem_src, target=mem_tgt)
            res = fallback.translate(clean_text)
            return res if res is not None else ""
        except Exception as fb_err:
            return f"Translation Service Error: {str(fb_err)}"
    except (LanguageNotSupportedException, TranslationNotFound) as e:
        return f"Translation Error: {str(e)}"
    except RequestError as e:
        return f"Network Error: {str(e)}"
    except Exception as e:
        return f"Unexpected Error: {str(e)}"

print("✅ Translation functions loaded successfully.")

✅ Translation functions loaded successfully.


## 6. Translation Testing
Let us test the required translation test cases specified in the internship guidelines:
1. **English → Telugu**: `"Hello, how are you?"`
2. **Telugu → English**: `"మీరు ఎలా ఉన్నారు?"`
3. **English → Hindi**: `"Good morning"`
4. **Hindi → English**: `"आप कैसे हैं?"`
5. Edge cases: Empty input, same source & target, stats.

In [6]:
# Test 1: English -> Telugu
text_en = "Hello, how are you?"
res_te = translate_text(text_en, source_language="English", target_language="Telugu")
print("Test 1: English -> Telugu")
print("Input :", text_en)
print("Output:", res_te)
print("Stats :", count_text_stats(text_en))
print("-" * 50)

Test 1: English -> Telugu
Input : Hello, how are you?
Output: హలో, ఎలా ఉన్నారు?
Stats : (19, 4)
--------------------------------------------------


In [7]:
# Test 2: Telugu -> English
text_te = "మీరు ఎలా ఉన్నారు?"
res_en = translate_text(text_te, source_language="Telugu", target_language="English")
print("Test 2: Telugu -> English")
print("Input :", text_te)
print("Output:", res_en)
print("-" * 50)

Test 2: Telugu -> English
Input : మీరు ఎలా ఉన్నారు?
Output: How are you?
--------------------------------------------------


In [8]:
# Test 3: English -> Hindi
text_gm = "Good morning"
res_hi = translate_text(text_gm, source_language="English", target_language="Hindi")
print("Test 3: English -> Hindi")
print("Input :", text_gm)
print("Output:", res_hi)
print("-" * 50)

Test 3: English -> Hindi
Input : Good morning
Output: शुभ प्रभात
--------------------------------------------------


In [9]:
# Test 4: Hindi -> English
text_hi = "आप कैसे हैं?"
res_en2 = translate_text(text_hi, source_language="Hindi", target_language="English")
print("Test 4: Hindi -> English")
print("Input :", text_hi)
print("Output:", res_en2)
print("-" * 50)

Test 4: Hindi -> English
Input : आप कैसे हैं?
Output: How are you?
--------------------------------------------------


In [10]:
# Test 5: Edge Cases & Validation
print("Test 5: Edge Cases")
print("Empty input:", repr(translate_text("")))
print("Whitespace only:", repr(translate_text("    ")))
print("Same source and target:", translate_text("Artificial Intelligence", "English", "English"))
print("Invalid language:", translate_text("Test", "Kryptonian", "English"))
print("Word & char count for 'Artificial Intelligence':", count_text_stats("Artificial Intelligence"))

Test 5: Edge Cases
Empty input: ''
Whitespace only: ''
Same source and target: Artificial Intelligence
Invalid language: Language Error: Unsupported language: 'Kryptonian'.
Word & char count for 'Artificial Intelligence': (23, 2)


## 7. Text-to-Speech (TTS) Integration
We implement `generate_tts_audio` using `gTTS` (Google Text-to-Speech) to generate audio pronunciation for translated text across all 15 supported languages.

In [11]:
def generate_tts_audio(text: str, target_language: str = "Telugu"):
    """
    Synthesizes speech from translated text using Google Text-to-Speech (gTTS).
    """
    if not text or not text.strip():
        return None

    clean_text = text.strip()
    if clean_text.startswith("Error:") or clean_text.startswith("Language Error:"):
        return None

    try:
        lang_code = get_language_code(target_language, is_source=False)
        tts = gTTS(text=clean_text, lang=lang_code, slow=False)
        temp_file = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        temp_path = temp_file.name
        temp_file.close()
        tts.save(temp_path)
        return temp_path
    except Exception as e:
        print(f"[TTS Warning]: {e}")
        return None

# Quick TTS test
test_audio = generate_tts_audio("నమస్కారం", "Telugu")
if test_audio and os.path.exists(test_audio):
    print("✅ TTS audio generated and verified successfully.")

✅ TTS audio generated and verified successfully.


## 8. Interactive Apple iOS Interface with Transition Flow
We construct the web interface designed with Apple Human Interface Guidelines:
- **Step 1**: Selected Language (From) & Translation Language (To) with Text Input Card at top.
- **Step 2**: Primary Apple Translate pill button.
- **Step 3**: Animated Spring Transition Result Card (revealed upon translation with speech, reverse, and clear actions).
- **Step 4**: Inset Grouped Translation History list below.

In [12]:
"""
AI Language Translation Tool
Application: app.py

Author: Saabeer
Project: LanguageTranslation
Description:
A refined, high-contrast Apple iOS-inspired Language Translation application
built with Python, Gradio, deep-translator, and Google Text-to-Speech (gTTS).
Features high-contrast text visibility, native Apple SF Pro typography,
and the requested sequential layout with animated spring transition.
"""

import inspect
from datetime import datetime
from typing import Tuple, List, Optional
import gradio as gr

from translation import (
    SUPPORTED_LANGUAGES,
    SOURCE_LANGUAGES,
    TARGET_LANGUAGES,
    translate_text,
    count_text_stats,
    generate_tts_audio
)


# -------------------------------------------------------------------------
# Apple iOS Theme Tokens (Gradio Theme Engine)
# Ensures 100% high-contrast readability in both Light and Dark mode.
# -------------------------------------------------------------------------
def get_ios_theme() -> gr.themes.Soft:
    """
    Constructs a native Apple iOS theme with high-contrast text and surfaces.
    """
    theme = gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate",
        radius_size="lg",
        text_size="md",
        font=[
            "-apple-system",
            "BlinkMacSystemFont",
            "SF Pro Text",
            "SF Pro Display",
            "Helvetica Neue",
            "sans-serif"
        ]
    ).set(
        # Canvas Background
        body_background_fill="#F2F2F7",
        body_background_fill_dark="#000000",
        
        # Inset Grouped Card Fill
        block_background_fill="#FFFFFF",
        block_background_fill_dark="#1C1C1E",
        block_border_width="1px",
        block_border_color="rgba(60, 60, 67, 0.15)",
        block_border_color_dark="rgba(255, 255, 255, 0.15)",
        block_radius="18px",
        block_shadow="0 4px 20px rgba(0, 0, 0, 0.04)",
        
        # Primary Action Button (Apple System Blue)
        button_primary_background_fill="#007AFF",
        button_primary_background_fill_hover="#0062CC",
        button_primary_text_color="#FFFFFF",
        
        # Secondary Action Buttons (Apple System Gray Fill)
        button_secondary_background_fill="#E5E5EA",
        button_secondary_background_fill_dark="#2C2C2E",
        button_secondary_text_color="#007AFF",
        button_secondary_text_color_dark="#0A84FF",
        
        # Form Inputs (High-Contrast White / Dark Fill)
        input_background_fill="#FFFFFF",
        input_background_fill_dark="#2C2C2E",
        input_border_color="rgba(60, 60, 67, 0.20)",
        input_border_color_dark="rgba(255, 255, 255, 0.20)",
        input_radius="14px"
    )
    return theme


# -------------------------------------------------------------------------
# Apple iOS Custom CSS (Enhancements & Transitions)
# -------------------------------------------------------------------------
APPLE_IOS_CSS = """
/* Container Layout */
.gradio-container {
    max-width: 820px !important;
    margin: 20px auto 40px auto !important;
    padding: 0 16px !important;
}

/* Apple Header Banner */
.apple-header {
    text-align: center;
    padding: 22px 16px;
    background: #FFFFFF;
    border-radius: 20px;
    border: 1px solid rgba(60, 60, 67, 0.12);
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.04);
    margin-bottom: 20px;
}

:is(.dark) .apple-header {
    background: #1C1C1E !important;
    border-color: rgba(255, 255, 255, 0.12) !important;
}

.apple-pill-tag {
    display: inline-block;
    background: rgba(0, 122, 255, 0.10);
    color: #007AFF;
    padding: 4px 14px;
    border-radius: 9999px;
    font-size: 0.78rem;
    font-weight: 700;
    letter-spacing: 0.4px;
    text-transform: uppercase;
    margin-bottom: 8px;
}

.apple-title {
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    letter-spacing: -0.6px !important;
    margin: 2px 0 6px 0 !important;
    line-height: 1.15;
}

.apple-subtitle {
    font-size: 0.96rem !important;
    color: #8E8E93 !important;
    margin: 0 !important;
}

/* Apple Primary Button (Full-Width Pill) */
.apple-translate-btn button {
    background: #007AFF !important;
    background-image: linear-gradient(180deg, #0A84FF 0%, #007AFF 100%) !important;
    color: #FFFFFF !important;
    border-radius: 9999px !important;
    font-size: 1.1rem !important;
    font-weight: 700 !important;
    letter-spacing: -0.2px !important;
    padding: 14px 30px !important;
    border: none !important;
    box-shadow: 0 6px 20px rgba(0, 122, 255, 0.35) !important;
    transition: all 0.15s ease !important;
    cursor: pointer !important;
    width: 100% !important;
}

.apple-translate-btn button:hover {
    background-image: linear-gradient(180deg, #007AFF 0%, #0062CC 100%) !important;
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 24px rgba(0, 122, 255, 0.42) !important;
}

.apple-translate-btn button:active {
    transform: scale(0.97) !important;
    box-shadow: 0 2px 8px rgba(0, 122, 255, 0.2) !important;
}

/* Result Card with Smooth Spring Transition */
@keyframes iosReveal {
    0% {
        opacity: 0;
        transform: translateY(18px);
    }
    100% {
        opacity: 1;
        transform: translateY(0);
    }
}

.ios-result-section {
    animation: iosReveal 0.35s cubic-bezier(0.16, 1, 0.3, 1) forwards !important;
    margin-top: 14px !important;
    margin-bottom: 20px !important;
    padding: 18px 20px !important;
    background: #FFFFFF !important;
    border-radius: 20px !important;
    border: 1.5px solid rgba(0, 122, 255, 0.25) !important;
    box-shadow: 0 6px 28px rgba(0, 122, 255, 0.08) !important;
}

:is(.dark) .ios-result-section {
    background: #1C1C1E !important;
    border-color: rgba(10, 132, 255, 0.35) !important;
}

.ios-result-tag {
    display: inline-block;
    background: rgba(0, 122, 255, 0.10);
    color: #007AFF;
    padding: 4px 12px;
    border-radius: 9999px;
    font-size: 0.8rem;
    font-weight: 700;
    margin-bottom: 8px;
}

/* Apple Secondary Pill Buttons */
button[variant="secondary"] {
    border-radius: 9999px !important;
    font-weight: 600 !important;
    transition: all 0.15s ease !important;
}

button[variant="secondary"]:active {
    transform: scale(0.97) !important;
}

/* Destructive Button */
button[variant="stop"] {
    border-radius: 9999px !important;
    font-weight: 600 !important;
}

/* Input Stats Pill */
.stats-chip {
    font-size: 0.82rem;
    font-weight: 600;
    color: #8E8E93;
    margin-top: 4px;
}

/* Footer */
.apple-footer {
    text-align: center;
    padding: 20px 0 8px 0;
    color: #8E8E93;
    font-size: 0.82rem;
}
"""


# -------------------------------------------------------------------------
# UI Event Handlers
# -------------------------------------------------------------------------
def on_translate(
    text: str,
    source_lang: str,
    target_lang: str,
    history: List[List[str]]
) -> Tuple[gr.update, str, str, Optional[str], str, List[List[str]], List[List[str]]]:
    """
    Translates input text and reveals the result section in a smooth transition.
    """
    if not text or not text.strip():
        stats = "📊 0 characters • 0 words"
        return gr.update(visible=False), "", stats, None, "", history, history

    # Perform real translation
    translated = translate_text(text, source_lang, target_lang)

    # Compute text metrics
    chars, words = count_text_stats(text)
    stats = f"📊 {chars} characters • {words} words"

    # Auto-generate speech audio if translation was successful
    audio_path = None
    if translated and not translated.startswith("Error:") and not translated.startswith("Language Error:"):
        audio_path = generate_tts_audio(translated, target_lang)

    # Result tag
    badge_label = f"✦ Translation Result ({target_lang})"

    # Update history log
    timestamp = datetime.now().strftime("%H:%M:%S")
    new_entry = [timestamp, source_lang, target_lang, text.strip(), translated]
    updated_history = [new_entry] + (history or [])

    # Reveal the result container smoothly
    return gr.update(visible=True), translated, stats, audio_path, badge_label, updated_history, updated_history


def on_swap_languages(
    source_lang: str,
    target_lang: str,
    input_text: str,
    output_text: str
) -> Tuple[str, str, str, str, str, Optional[str], gr.update]:
    """
    Swaps source and target languages, reversing input/output for instant reverse-translation.
    """
    new_source = target_lang
    new_target = "English" if source_lang == "Auto Detect" else source_lang

    new_input = (
        output_text
        if output_text and not output_text.startswith("Error:") and not output_text.startswith("Language Error:")
        else input_text
    )
    new_output = ""

    chars, words = count_text_stats(new_input)
    stats = f"📊 {chars} characters • {words} words"

    # Hide result card until user clicks Translate again
    return new_source, new_target, new_input, new_output, stats, None, gr.update(visible=False)


def on_clear() -> Tuple[gr.update, str, str, str, Optional[str]]:
    """
    Resets input, output, stats, audio, and smoothly collapses the result card.
    """
    return gr.update(visible=False), "", "", "📊 0 characters • 0 words", None


def on_listen(output_text: str, target_lang: str) -> Optional[str]:
    """
    Generates voice playback on demand.
    """
    if not output_text or not output_text.strip():
        return None
    return generate_tts_audio(output_text, target_lang)


def on_input_text_change(text: str) -> str:
    """
    Dynamically recalculates live character and word counters.
    """
    chars, words = count_text_stats(text)
    return f"📊 {chars} characters • {words} words"


def on_clear_history() -> Tuple[List[List[str]], List[List[str]]]:
    """
    Clears translation history records.
    """
    return [], []


# -------------------------------------------------------------------------
# Application Builder
# -------------------------------------------------------------------------
def create_app() -> gr.Blocks:
    """
    Constructs the streamlined Apple iOS translation interface.
    Layout order:
    1. Language selectors & Text input box (at top)
    2. Primary Translate button (under that)
    3. Animated Transition Result card (revealed on translate)
    4. Inset Grouped History section (under that)
    """
    source_choices = list(SOURCE_LANGUAGES.keys())
    target_choices = list(TARGET_LANGUAGES.keys())

    # Detect copy button parameter across Gradio versions
    tb_sig = inspect.signature(gr.Textbox.__init__).parameters
    copy_kwarg = {}
    if "buttons" in tb_sig:
        copy_kwarg = {"buttons": ["copy"]}
    elif "show_copy_button" in tb_sig:
        copy_kwarg = {"show_copy_button": True}

    with gr.Blocks(title="Translate — Apple iOS Experience") as demo:
        # History state management
        history_state = gr.State([])

        # 1. Navigation Header
        gr.HTML("""
        <div class="apple-header">
            <span class="apple-pill-tag">✦ Neural Engine</span>
            <h1 class="apple-title">Translate</h1>
            <p class="apple-subtitle">Instant speech and text translation across 15+ languages</p>
        </div>
        """)

        # 2. Main Input Card: Language Selectors & Text Input Box
        with gr.Group():
            with gr.Row():
                source_dropdown = gr.Dropdown(
                    choices=source_choices,
                    value="English",
                    label="Selected Language (From)",
                    info="Choose input language or Auto Detect"
                )
                target_dropdown = gr.Dropdown(
                    choices=target_choices,
                    value="Telugu",
                    label="Translation Language (To)",
                    info="Choose destination language"
                )

            input_text = gr.Textbox(
                label="Source Text",
                placeholder="Type or paste text to translate here...",
                lines=4,
                max_lines=8,
                **copy_kwarg
            )
            stats_display = gr.Markdown(
                "📊 0 characters • 0 words",
                elem_classes=["stats-chip"]
            )

        # 3. Under that: Primary Translate Button
        with gr.Row(elem_classes=["apple-translate-btn"]):
            translate_btn = gr.Button(
                "🚀 Translate",
                variant="primary"
            )

        # 4. Under that: Animated Transition Result Card (Revealed on Translate)
        with gr.Column(visible=False, elem_classes=["ios-result-section"]) as result_card:
            result_badge = gr.Markdown(
                "✦ Translation Result",
                elem_classes=["ios-result-tag"]
            )
            output_text = gr.Textbox(
                label="Translated Text",
                placeholder="Translation appears here...",
                lines=4,
                max_lines=8,
                interactive=False,
                **copy_kwarg
            )
            audio_output = gr.Audio(
                label="🔊 Speech Pronunciation",
                type="filepath",
                interactive=False
            )
            # Result Action Controls
            with gr.Row():
                swap_btn = gr.Button("⇄ Swap Languages", variant="secondary")
                listen_btn = gr.Button("🔊 Re-Play Speech", variant="secondary")
                clear_btn = gr.Button("✕ Clear Text", variant="secondary")

        # 5. Under that: Inset Grouped History Section
        with gr.Accordion("Recent Translations", open=True):
            history_table = gr.Dataframe(
                headers=["Time", "From", "To", "Original Text", "Translated Text"],
                datatype=["str", "str", "str", "str", "str"],
                value=[],
                interactive=False,
                wrap=True
            )
            with gr.Row():
                clear_history_btn = gr.Button("Clear History", size="sm", variant="stop")

        # 6. Apple Footer
        gr.HTML(
            """
            <div class="apple-footer">
                Designed with Apple Human Interface Guidelines • Neural Speech & Translation Engine
            </div>
            """
        )

        # -----------------------------------------------------------------
        # Event Bindings
        # -----------------------------------------------------------------
        # Dynamic stats on input text change
        input_text.change(
            fn=on_input_text_change,
            inputs=[input_text],
            outputs=[stats_display]
        )

        # Translate button click
        translate_btn.click(
            fn=on_translate,
            inputs=[input_text, source_dropdown, target_dropdown, history_state],
            outputs=[result_card, output_text, stats_display, audio_output, result_badge, history_table, history_state]
        )

        # Enter key submit also triggers translation
        input_text.submit(
            fn=on_translate,
            inputs=[input_text, source_dropdown, target_dropdown, history_state],
            outputs=[result_card, output_text, stats_display, audio_output, result_badge, history_table, history_state]
        )

        # Swap languages click
        swap_btn.click(
            fn=on_swap_languages,
            inputs=[source_dropdown, target_dropdown, input_text, output_text],
            outputs=[source_dropdown, target_dropdown, input_text, output_text, stats_display, audio_output, result_card]
        )

        # Clear button click
        clear_btn.click(
            fn=on_clear,
            inputs=[],
            outputs=[result_card, input_text, output_text, stats_display, audio_output]
        )

        # Listen button click
        listen_btn.click(
            fn=on_listen,
            inputs=[output_text, target_dropdown],
            outputs=[audio_output]
        )

        # Clear history click
        clear_history_btn.click(
            fn=on_clear_history,
            inputs=[],
            outputs=[history_table, history_state]
        )

    return demo


# -------------------------------------------------------------------------
# Application Entry Point
# -------------------------------------------------------------------------
if __name__ == "__main__":
    app = create_app()
    print("Launching Apple iOS-Style AI Language Translation Tool...")
    theme = get_ios_theme()
    app.launch(theme=theme, css=APPLE_IOS_CSS, share=False)


✅ Gradio application constructed and ready to launch.


## 9. Launch Application
Run the cell below to launch the web application. In Google Colab, `share=True` creates a temporary public URL for easy browser testing and video demonstration.

In [ ]:
# In Google Colab, share=True creates a public Gradio link
# Set share=False when running locally
demo.launch(share=True)